In [ ]:
import json
import pandas as pd
import datetime
import xarray as xr

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np


def preprocess(ds):
    # Remove batch dimension if it exists
    if 'batch' in ds.dims:
        ds = ds.squeeze(dim="batch")
    return ds


def preprocess_era5_dataset(ds):
    ds = ds.rename({"valid_time": "datetime", "latitude": "lat", "longitude": "lon"})
    ds = ds.set_index({"datetime": "datetime"})
    ds = ds.reindex(lat=ds.lat[::-1])  # Reverse latitude order to match Graphcast data
    ds = ds.drop_vars(["number", "expver"])
    return ds

def average_area(start_date, end_date, tod, da):
    da = da.sel(lat=slice(5, 14), lon=slice(35, 43)).sel(datetime=slice(start_date, end_date))
    da = da.sel(datetime=da.datetime.dt.hour == tod)
    return da.mean(dim=["lat", "lon"])

In [ ]:
forecast_data = xr.open_mfdataset(
    [f"/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/graphcast_full/graphcast_predictions/pred_2016{day.strftime('%m%d')}-06_n0.nc" for day in pd.date_range("2016-01-01", "2016-04-01")],
    concat_dim="datetime",
    combine="nested",
    preprocess=preprocess,
).rename({"time": "datetime"})
forecast_data = forecast_data.set_index({"datetime": "datetime"})

analysis_data = preprocess_era5_dataset(xr.open_dataset("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/era5_data/localised_2mt_2016.nc"))

forecast_timeseries = average_area("2016-01-01", "2016-04-01", 6, forecast_data["2m_temperature"]).load()
analysis_timeseries = average_area("2016-01-01", "2016-04-01", 6, analysis_data["t2m"])

error_dates = pd.read_csv("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/error_dates_2014_15_16.csv")
error_dates['datetime'] = pd.to_datetime(error_dates['datetime'])


# METAR data

## Loading and preprocessing

In [ ]:
with open("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/ground_obs/metar2016.json", "r") as f:
    metar_data = json.load(f)


In [ ]:
def process_message(message):
    year      = message[-1][0][2][1][0]["value"]
    month     = message[-1][0][2][1][1][0]["value"]
    day       = message[-1][0][2][1][1][1][0]["value"]
    hour      = message[-1][0][2][1][1][1][1][0]["value"]
    min       = message[-1][0][2][1][1][1][1][1][0]["value"]
    lat       = message[-1][0][2][1][1][1][1][1][1][0]["value"]
    lon       = message[-1][0][2][1][1][1][1][1][1][1][0]["value"]
    air_temp  = message[-1][0][2][1][1][1][1][1][1][1][1][2][1]["value"]
    station_h = message[-1][0][2][1][1][1][1][1][1][1][1][0]["value"]

    date = datetime.datetime(year, month, day, hour, min)
    return date, lat, lon, air_temp, station_h

# year = metar_data["messages"][0][-1][0][2][1][0]
# month = metar_data["messages"][0][-1][0][2][1][1][0]
# day = metar_data["messages"][0][-1][0][2][1][1][1][0]
# hour = metar_data["messages"][0][-1][0][2][1][1][1][1][0]
# min = metar_data["messages"][0][-1][0][2][1][1][1][1][1][0]
# lat = metar_data["messages"][0][-1][0][2][1][1][1][1][1][1][0]
# lon = metar_data["messages"][0][-1][0][2][1][1][1][1][1][1][1][0]
# hos = metar_data["messages"][0][-1][0][2][1][1][1][1][1][1][1][1][0] # height of station
# wind_data = metar_data["messages"][0][-1][0][2][1][1][1][1][1][1][1][1][1]
# has = wind_data[0] # height above station
# wind_dir = wind_data[1]

In [ ]:
processed = [process_message(message) for message in metar_data["messages"]] 
metar_df = pd.DataFrame(processed, columns=["date", "lat", "lon", "air_temp", "station_h"])

In [ ]:
metar_df.to_csv("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/ground_obs/METAR_processed.csv")

## Plot map of all stations

In [ ]:

proj = ccrs.PlateCarree()  # simple equirectangular

fig, ax = plt.subplots(figsize=(8,5), subplot_kw={'projection': proj})

# Add features
ax.add_feature(cfeature.COASTLINE, linewidth=0.6)
ax.add_feature(cfeature.BORDERS, linewidth=0.4)
# ax.add_feature(cfeature.LAND, facecolor='#f0f0e8')
# ax.add_feature(cfeature.OCEAN, facecolor='#e0f4ff')

# Scatter points
sc = ax.scatter(
    metar_df['lon'], metar_df['lat'],
    c=metar_df['air_temp'],            # color by variable
    cmap='coolwarm',
    s=40,                        # point size
    edgecolor='k',
    linewidth=0.5,
    transform=ccrs.PlateCarree()
)

# Colorbar
cb = fig.colorbar(sc, ax=ax, shrink=0.75, pad=0.02)
cb.set_label('Air temperature (K)')

ax.set_title('Station Air Temperature')
plt.tight_layout()
plt.show()

## Look for the station at Addis Ababa airport

In [ ]:
latrange = (5,15)
lonrange = (32,42)

metar_roi = metar_df[(metar_df['lat'].between(*latrange)) & (metar_df['lon'].between(*lonrange))]

metar_roi.lat.unique(), metar_roi.lon.unique() # only 1 station

In [ ]:
plt.plot(metar_roi["date"], metar_roi["air_temp"], label="Air temp. obs. 8.98N, 38.79E", color="blue")
# plt.plot(forecast_timeseries.datetime, forecast_timeseries, label="Forecast 2mt", color="orange")
# plt.plot(analysis_timeseries["datetime"], analysis_timeseries, label="Analysis 2mt", color="green")

plt.plot(forecast_data.datetime, forecast_data["2m_temperature"].sel(lat=8.98, lon=38.79, method="nearest"), label="Forecast 2mt", color="orange")
plt.plot(analysis_data.datetime, analysis_data["t2m"].sel(lat=8.98, lon=38.79, method="nearest"), label="Analysis 2mt", color="green")


metar_roi.date[metar_roi["air_temp"].isna()]

for date in error_dates["datetime"][error_dates["datetime"].isin(metar_roi["date"])]:
    plt.vlines(date, metar_roi["air_temp"].min(), metar_roi["air_temp"].max(), color='red', linestyle='--')
    
plt.ylabel("Temperature")
plt.legend()

plt.xlim(np.datetime64("2016-01-01"), np.datetime64("2016-04-01"))

In [ ]:
proj = ccrs.PlateCarree()  # simple equirectangular

fig, ax = plt.subplots(figsize=(8,5), subplot_kw={'projection': proj})

# Add features
ax.add_feature(cfeature.COASTLINE, linewidth=0.6)
ax.add_feature(cfeature.BORDERS, linewidth=0.4)
# ax.add_feature(cfeature.LAND, facecolor='#f0f0e8')
# ax.add_feature(cfeature.OCEAN, facecolor='#e0f4ff')

# Scatter points
sc = ax.scatter(
    metar_df['lon'], metar_df['lat'],
    c=metar_df['air_temp'],            # color by variable
    cmap='coolwarm',
    s=40,                        # point size
    edgecolor='k',
    linewidth=0.5,
    transform=ccrs.PlateCarree()
)

ax.set_extent([30, 50, 0, 20], crs=ccrs.PlateCarree())



# Colorbar
cb = fig.colorbar(sc, ax=ax, shrink=0.75, pad=0.02)
cb.set_label('Air temperature (°C)' if df['air_temp'].max() < 100 else 'Air temperature (K)')

# Optional: focus extent (lon_min, lon_max, lat_min, lat_max)
# ax.set_extent([-30, 40, 30, 65], crs=ccrs.PlateCarree())

ax.set_title('Station Air Temperature')
plt.tight_layout()
plt.show()

# Synop auto data

## Loading the data

In [ ]:
with open("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/ground_obs/synop_auto_land.json", "r") as f:
    synop_auto = json.load(f)

In [ ]:
def process_synop_message(message):
    data = message[-1][0][2][1][1]
    year = data[0]["value"]
    month = data[1][0]["value"]
    day = data[1][1][0]["value"]
    hour = data[1][1][1][0]["value"]
    min = data[1][1][1][1][0]["value"]
    lat = data[1][1][1][1][1][0]["value"]
    lon = data[1][1][1][1][1][1][0]["value"]
    min_ground_t_12h = data[1][1][1][1][1][1][1][1]["value"]
    min_t2m_12h =      data[1][1][1][1][1][1][1][2]["value"]
    max_t2m_24h =      data[1][1][1][1][1][1][1][3]["value"]

    date = datetime.datetime(year, month, day, hour, min)

    return date, lat, lon, min_ground_t_12h, min_t2m_12h, max_t2m_24h

In [ ]:
processed_synop = [process_synop_message(message) for message in synop_auto["messages"]] 

synop_auto_df = pd.DataFrame(processed_synop, columns=["date", "lat", "lon", "min_ground_t_12h", "min_t2m_12h", "max_t2m_24h"])
synop_auto_df

In [ ]:
synop_auto_df[ (synop_auto_df['date'] == datetime.datetime(2016,1,18,6))  & (synop_auto_df['lat'].between(-3,18)) & (synop_auto_df['lon'].between(30,50)) ].to_csv('synop_auto_20160118_06.csv', index=False)

## How much data do we have? What locations do we have it for?

In [ ]:
latrange = (5,14)
lonrange = (35,43)

roi = synop_auto_df[(synop_auto_df['lat'].between(*latrange)) & (synop_auto_df['lon'].between(*lonrange))]

min_t2m_12h_stations = roi[["lat", "lon", "min_t2m_12h", "date"]][roi["min_t2m_12h"].notna()]
max_t2m_24h_stations = roi[["lat", "lon", "max_t2m_24h", "date"]][roi["max_t2m_24h"].notna()] # only has 3 entries

stations = min_t2m_12h_stations[["lat", "lon"]].drop_duplicates().reset_index(drop=True)

daily_missing = roi.groupby("date").apply(lambda x: x["min_t2m_12h"].isna().sum()).reset_index(name="missing_stations")

error_dates_in_period = error_dates["datetime"][(error_dates["datetime"] < "2016-04-01") &( error_dates["datetime"] > "2016-01-01")]
error_dates_with_data = error_dates_in_period[error_dates_in_period.isin(roi["date"])]
error_dates_without_data = error_dates_in_period[~error_dates_in_period.isin(roi["date"])]

len(error_dates_with_data), len(error_dates_without_data)

print(f"We only have data for {len(roi['date'].unique())} days over 3 months")
print(f"There are {len(stations)} stations with data. The maximum number of missing stations in a day is {daily_missing['missing_stations'].max()}.")

print(f"There are a total of {len(error_dates_in_period)} error dates in the period, of which {len(error_dates_with_data)} have data and {len(error_dates_without_data)} do not.")

In [ ]:
for station in stations.itertuples(index=False):
    this_station = roi[(roi["lat"] == station.lat) & (roi["lon"] == station.lon)]
    plt.plot(this_station["date"], this_station["min_t2m_12h"], label=f"{station.lat}, {station.lon}", alpha=0.3)
    plt.plot(roi["date"].unique(), roi.groupby("date").mean()["min_t2m_12h"], label="Mean", color="black")

# for date in error_dates["datetime"][(error_dates["datetime"] < "2016-04-01") &( error_dates["datetime"] > "2016-01-01")]:
#     plt.vlines(date, roi["min_t2m_12h"].min(), roi["min_t2m_12h"].max(), color='red', linestyle='--')

plt.ylabel("Minimum 2mt recorded in prev. 12h (K)")
plt.title("Averaged minimum 2mt recorded in previous 12h at stations in ROI")


In [ ]:
proj = ccrs.PlateCarree()

fig, ax = plt.subplots(figsize=(8,5), subplot_kw={'projection': proj})

ax.add_feature(cfeature.COASTLINE, linewidth=0.6)
ax.add_feature(cfeature.BORDERS, linewidth=0.4)

# Scatter points
sc = ax.scatter(
    roi['lon'], roi['lat'],
    c=roi['min_t2m_12h'],
    cmap='coolwarm',
    s=40,
    edgecolor='k',
    linewidth=0.5,
    transform=ccrs.PlateCarree()
)

ax.set_extent([35, 43, 5, 14], crs=ccrs.PlateCarree())

ax.set_title('Station Air Temperature')
plt.tight_layout()
plt.show()

# Synop record

In [ ]:
with open("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/ground_obs/synop_record_2_land_file.json", "r") as f:
    synop_record = json.load(f)

In [ ]:
processed_synop_record = [process_synop_message(message) for message in synop_record["messages"]]

synop_record_df = pd.DataFrame(processed_synop_record, columns=["date", "lat", "lon", "min_ground_t_12h", "min_t2m_12h", "max_t2m_24h"])

synop_record_df

In [ ]:
synop_record_df[ (synop_record_df['date'] == datetime.datetime(2016,1,18,6))  & (synop_record_df['lat'].between(-3,18)) & (synop_record_df['lon'].between(30,50)) ]

In [ ]:
proj = ccrs.PlateCarree()  # simple equirectangular

fig, ax = plt.subplots(figsize=(8,5), subplot_kw={'projection': proj})

# Add features
ax.add_feature(cfeature.COASTLINE, linewidth=0.6)
ax.add_feature(cfeature.BORDERS, linewidth=0.4)
# ax.add_feature(cfeature.LAND, facecolor='#f0f0e8')
# ax.add_feature(cfeature.OCEAN, facecolor='#e0f4ff')

# Scatter points
sc = ax.scatter(
    synop_record_df['lon'], synop_record_df['lat'],
    c=synop_record_df['min_t2m_12h'],            # color by variable
    cmap='coolwarm',
    s=40,                        # point size
    edgecolor='k',
    linewidth=0.5,
    transform=ccrs.PlateCarree()
)

# Colorbar
cb = fig.colorbar(sc, ax=ax, shrink=0.75, pad=0.02)
cb.set_label('Air temperature (K)')

ax.set_title('Station Air Temperature')
plt.tight_layout()
plt.show()

In [ ]:
latrange = (5,14)
lonrange = (35,43)

roi = synop_record_df[(synop_record_df['lat'].between(*latrange)) & (synop_record_df['lon'].between(*lonrange))]

min_t2m_12h_stations = roi[["lat", "lon", "min_t2m_12h", "date"]][roi["min_t2m_12h"].notna()]
max_t2m_24h_stations = roi[["lat", "lon", "max_t2m_24h", "date"]][roi["max_t2m_24h"].notna()] # only has 3 entries

stations = min_t2m_12h_stations[["lat", "lon"]].drop_duplicates().reset_index(drop=True)

daily_missing = roi.groupby("date").apply(lambda x: x["min_t2m_12h"].isna().sum()).reset_index(name="missing_stations")

error_dates_in_period = error_dates["datetime"][(error_dates["datetime"] < "2016-04-01") &( error_dates["datetime"] > "2016-01-01")]
error_dates_with_data = error_dates_in_period[error_dates_in_period.isin(roi["date"])]
error_dates_without_data = error_dates_in_period[~error_dates_in_period.isin(roi["date"])]

len(error_dates_with_data), len(error_dates_without_data)

print(f"We only have data for {len(roi['date'].unique())} days over 3 months")
print(f"There are {len(stations)} stations with data. The maximum number of missing stations in a day is {daily_missing['missing_stations'].max()}.")

print(f"There are a total of {len(error_dates_in_period)} error dates in the period, of which {len(error_dates_with_data)} have data and {len(error_dates_without_data)} do not.")

In [ ]:
for station in stations.itertuples(index=False):
    this_station = roi[(roi["lat"] == station.lat) & (roi["lon"] == station.lon)]
    plt.plot(this_station["date"], this_station["min_t2m_12h"], label=f"{station.lat}, {station.lon}", alpha=0.3)
    plt.plot(roi["date"].unique(), roi.groupby("date").mean()["min_t2m_12h"], label="Mean", color="black")

# for date in error_dates["datetime"][(error_dates["datetime"] < "2016-04-01") &( error_dates["datetime"] > "2016-01-01")]:
#     plt.vlines(date, roi["min_t2m_12h"].min(), roi["min_t2m_12h"].max(), color='red', linestyle='--')

plt.ylabel("Minimum 2mt recorded in prev. 12h (K)")
plt.title("Averaged minimum 2mt recorded in previous 12h at stations in ROI")


# Recreating the ERA5 method for rejecting observations, using METAR data

1. Work out the interpolated increment $\Delta X_i$ at the observation location $i$.
    - Increment = forecasted value at this location - previous ERA5 value.
    - Interpolation method? Probably just linear for now
    - Work out how to apply lapse rate correction
2. See if condition is met

In [ ]:
error_data = xr.open_mfdataset(
    [f"/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/graphcast_full/graphcast_predictions/errors_2016{day.strftime('%m%d-%H')}_n0.nc" for day in pd.date_range("2016-01-01", "2016-12-31", freq="6h")],
    concat_dim="datetime",
    combine="nested",
    preprocess=preprocess,
).drop_vars(["number", "batch"])

forecast_data = xr.open_mfdataset(
    [f"/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/graphcast_full/graphcast_predictions/pred_2016{day.strftime('%m%d-%H')}_n0.nc" for day in pd.date_range("2016-01-01", "2016-12-31", freq="6h")],
    concat_dim="datetime",
    combine="nested",
    preprocess=preprocess,
).rename({"time": "datetime"})
forecast_data = forecast_data.set_index({"datetime": "datetime"}).drop_vars(["number", "batch"])

analysis_data = forecast_data - error_data


def preprocess_ifs_data(ds):
    ds = ds.rename({
        'latitude': 'lat',
        'longitude': 'lon',
        'time': 'datetime',
    })
    ds = ds.reindex(lat = ds.lat[::-1])
    return ds
ifs_forecast = preprocess_ifs_data(xr.open_dataset("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/ifs_data/da_forecasts2016.nc"))
ifs_forecast = ifs_forecast.rename_vars({"t2m": "2m_temperature"})

error_dates = pd.read_csv("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/error_dates_2014_15_16.csv.csv")
error_dates['datetime'] = pd.to_datetime(error_dates['datetime'])
error_dates = error_dates["datetime"]


## Interpolating 2mt IFS forecast to the measurement loc using the lapse correction

In [ ]:
obs_loc = (8.98, 38.79) # lat, lon


geop = preprocess_era5_dataset(xr.load_dataset("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/era5_data/geop.nc"))

height = geop["z"]/9.81

station_height = 2334

lapse_correction = 4.5 * (height - station_height)/1000

height.sel(lat=obs_loc[0], lon=obs_loc[1], method="nearest").drop_vars("datetime")

lapse_correction = lapse_correction.coarsen(lon=4).mean()
lapse_correction = lapse_correction.coarsen(lat=4, boundary="pad").mean()
lapse_correction = lapse_correction.isel(datetime=0).drop_vars("datetime")

corrected_ifs = ifs_forecast["2m_temperature"] + lapse_correction.values

corrected_ifs = corrected_ifs.to_dataset()

## Loading observation data (METAR)

In [ ]:

metar_df = pd.read_csv("/network/group/aopp/predict/HMC007_CHRISTENSEN_GRAPHCAS/ground_obs/METAR_processed.csv")

metar_df['date'] = pd.to_datetime(metar_df['date'])

latrange = (5,15)
lonrange = (32,42)

metar_roi = metar_df[(metar_df['lat'].between(*latrange)) & (metar_df['lon'].between(*lonrange))]


metar_roi = metar_roi.set_index("date")
metar_roi = metar_roi.reindex(pd.date_range(start=metar_roi.index.min(), end=metar_roi.index.max(), freq='D'))

metar_roi


In [ ]:
metar_df[ (metar_df['date'] == datetime.datetime(2016,1,18,6))  & (metar_df['lat'].between(-3,18)) & (metar_df['lon'].between(30,50)) ].to_csv('metar_20160118_06.csv', index=False)

## Calculating the increment

In [ ]:
forecast_at_obs = corrected_ifs.interp(lat=obs_loc[0], lon=obs_loc[1], method="linear")["2m_temperature"]

mutual_dates = np.intersect1d(forecast_at_obs.datetime, metar_roi.index)

forecast_at_obs = forecast_at_obs[np.isin(forecast_at_obs.datetime, mutual_dates)]
metar_roi = metar_roi[metar_roi.index.isin(mutual_dates)]

Delta_X2 = metar_roi["air_temp"] - forecast_at_obs


In [ ]:
plt.plot(mutual_dates, Delta_X2, label="Interpolated Increment")

plt.vlines(error_dates, Delta_X2.min(), Delta_X2.max(), color='r', linestyles='--', label="Error Dates")

plt.xlim(np.datetime64("2016-01-01"), np.datetime64("2016-03-31"))

plt.ylabel("Estimated increment (K)")

In [ ]:
# midnight_analysis = analysis_data.shift(datetime=1)
# midnight_analysis = midnight_analysis.sel(datetime=midnight_analysis.datetime.dt.hour == 6)

# corrected_ifs = ifs_forecast + lapse_correction

# increment_6hr = ifs_forecast - midnight_analysis


# Delta_X = increment_6hr.interp(lat=obs_loc[0], lon=obs_loc[1], method="linear")["2m_temperature"].load()

# plt.plot(Delta_X.datetime, Delta_X, label="Interpolated Increment")

# plt.vlines(error_dates, 0, 12, color='r', linestyles='--', label="Error Dates")

# plt.xlim(np.datetime64("2016-01-01"), np.datetime64("2016-07-31"))
